In [1]:
import os, sys, json
import numpy as np
import mdtraj as md

from openmm.app import *
from openmm import *
from openmm.unit import *

In [2]:
def write_structure(sim: Simulation, pdb_fn: str):
    with open(pdb_fn, 'w') as f:
        PDBFile.writeFile(simulation.topology, simulation.context.getState(getPositions=True).getPositions(), f)
    print(f'Wrote: {pdb_fn}')

In [3]:
def get_positions_from_pdb(fname_pdb, fname_prmtop):
    c = md.load(fname_pdb, top=fname_prmtop)
    return c.xyz[0, :, :]

def run_nvt(initial_pdb_fn, prmtop_fn, std_out_fn, dcd_out_fn,
            dt=2.0, temp=300.0, n_total=5000000000, n_dcd=5000, n_stdout=5000):

    #Initialize OpenMM AMBER Prmtop
    prmtop = AmberPrmtopFile(prmtop_fn)
    
    #Create Gas phase system from prmtop
    system = prmtop.createSystem(nonbondedMethod=NoCutoff, constraints=None)
    integrator = LangevinIntegrator(temp*kelvin, 1.0/picosecond, dt*femtoseconds)
    
    #Try to use both GPU platforms and fall back on CPU if neither work
    try: #TRY opencl first
        print('OPENCL')
        platform = Platform.getPlatformByName('OpenCL')
        properties = {'OpenCLPrecision': 'mixed'}
        simulation = Simulation(prmtop.topology, system, integrator, platform, properties)
    except:
        try: #First fallback to CUDA
            print('CUDA')
            platform = Platform.getPlatformByName('CUDA')
            properties = {'CudaPrecision': 'mixed'}
            simulation = Simulation(prmtop.topology, system, integrator, platform, properties)
        except: #Final Fallback to cpu
            print('CPU')
            simulation = Simulation(prmtop.topology, system, integrator)
    
    #Set initial coordinates to the pdb file provided
    simulation.context.setPositions(get_positions_from_pdb(initial_pdb_fn, prmtop_fn))# nm to nm
    
    #Set a random set of initial velocities based on temperature
    simulation.context.setVelocitiesToTemperature(temp*kelvin)
    
    #Report status of simulation to text file (std_out_fn) every (n_stdout) steps
    SDR = StateDataReporter(std_out_fn, n_stdout, step=True, time=True,
                            potentialEnergy=True, temperature=True, remainingTime=True,
                            totalSteps=n_total, separator='   ::   ')
    simulation.reporters.append(SDR)
    
    #Write the coordinates to DCD file (dcd_out_fn) every (n_dcd) steps (default 10ps)
    DCR = DCDReporter(dcd_out_fn, n_dcd)
    simulation.reporters.append(DCR)
    
    #Run the simulation
    simulation.step(n_total)        

In [ ]:
initial_pdb_fn = 'Simulation/ala_deca_peptide.pdb'
prmtop_fn = 'Simulation/ala_deca_peptide.prmtop'
std_out_fn = 'Simulation/DA_LONG.stdout'
dcd_out_fn = 'Simulation/DA_LONG.dcd'

run_nvt(initial_pdb_fn, prmtop_fn, std_out_fn, dcd_out_fn)

OPENCL
